# 4. Multi-agente con LangGraph (lo que se usa ahora)

## Objetivos
- Distinguir **un agente con pasos** de un **equipo de verdad**.
- Implementar las tres topologías que aparecen en producción en 2026:
  1. **Supervisor** — un nodo elige la ruta.
  2. **Handoff** — un agente cede el control (`Command(goto=...)`). Es el recambio de OpenAI Swarm (archivada).
  3. **Fan-out** — el mismo nodo en paralelo (`Send`). Map-reduce, no “reunión”.
- Contrastar con CrewAI (roles) y con Python puro (algoritmo).
- Salir con una recomendación de proyecto medible (tokens, no feeling).

Lectura que cierra la sesión: [`4-tendencias-multiagente.md`](4-tendencias-multiagente.md).
El supervisor largo ya está en `3-langgraph-orquestacion.py`. Aquí caben los tres patrones, cortos.


### 1. Instalación y modelo rápido

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain-groq groq langgraph langchain python-dotenv
else:
    print("Entorno local: las dependencias ya las instaló uv sync.")


In [ ]:
import os
from typing import Annotated, Literal, TypedDict
import operator

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send

try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# Tres patrones × varias llamadas: modelo rápido a propósito.
llm = ChatGroq(
    model=os.getenv("GROQ_MODEL_FAST", "openai/gpt-oss-20b"),
    temperature=0,
    reasoning_effort="low",
)


def mostrar_grafo(app):
    g = app.get_graph()
    print("Nodos:", ", ".join(n.name for n in g.nodes.values()))
    for e in g.edges:
        marca = "  (condicional)" if getattr(e, "conditional", False) else ""
        print(f"  {e.source} → {e.target}{marca}")


print("✅ LLM rápido listo:", llm.model_name)


### 2. La pregunta de mercado (antes del código)

Un equipo de agentes **no es más profesional**. Es más caro y más opaco. En clase (y en una entrevista) la respuesta correcta es:

> Empiezo con un agente. Añado un segundo cerebro solo si los prompts o los permisos chocan.

Eso está desarrollado, con costos, en `orchestration-guide.md` y en `4-tendencias-multiagente.md`.


### 3. Supervisor — orquestación clásica, grafo actual

Un nodo **no resuelve** la tarea: solo elige. Los especialistas no se hablan. Fácil de auditar (“¿quién contestó?”).

Es el mismo patrón que `3-langgraph-orquestacion.py`, compactado a dos rutas.


In [ ]:
class EstadoSup(TypedDict):
    consulta: str
    ruta: str
    respuesta: str


def supervisar(state: EstadoSup) -> dict:
    crudo = llm.invoke([
        SystemMessage(content="Una palabra: legal o producto. Sin explicación."),
        HumanMessage(content=state["consulta"]),
    ]).content
    ruta = "legal" if "legal" in (crudo or "").lower() else "producto"
    print(f"🧭 Supervisor → {ruta} ({crudo!r})")
    return {"ruta": ruta}


def legal(state: EstadoSup) -> dict:
    texto = llm.invoke([
        SystemMessage(content="Abogado en Chile. 3 líneas, riesgo y qué no prometas. No cites RGPD ni LOPD española."),
        HumanMessage(content=state["consulta"]),
    ]).content
    return {"respuesta": texto}


def producto(state: EstadoSup) -> dict:
    texto = llm.invoke([
        SystemMessage(content="PM. 3 líneas, alcance y next step."),
        HumanMessage(content=state["consulta"]),
    ]).content
    return {"respuesta": texto}


def ruta_sup(state: EstadoSup) -> Literal["legal", "producto"]:
    return "legal" if state["ruta"] == "legal" else "producto"


g = StateGraph(EstadoSup)
g.add_node("supervisar", supervisar)
g.add_node("legal", legal)
g.add_node("producto", producto)
g.add_edge(START, "supervisar")
g.add_conditional_edges("supervisar", ruta_sup)
g.add_edge("legal", END)
g.add_edge("producto", END)
app_sup = g.compile()
mostrar_grafo(app_sup)

out = app_sup.invoke({
    "consulta": "¿Podemos guardar las notas de los alumnos en un Excel compartido?",
    "ruta": "",
    "respuesta": "",
})
print("\nRespuesta:\n", out["respuesta"])


### 4. Handoff — el recambio de Swarm

En un supervisor, A no le habla a B. En un **handoff**, A termina su turno y **pasa el control** (`Command(goto="b")`). B ve las notas. Eso es lo que hacía OpenAI Swarm; la librería se archivó, el patrón no.

Tope duro: si B quiere devolver a A, un contador. Sin tope, dos agentes se facturan.


In [ ]:
class EstadoHand(TypedDict):
    consulta: str
    notas: str
    saltos: int
    respuesta: str


MAX_SALTOS = 2


def recepcion(state: EstadoHand) -> Command:
    notas = llm.invoke([
        SystemMessage(content="Recepcionista. En 1 frase: ¿falta un dato legal o de producto?"),
        HumanMessage(content=state["consulta"]),
    ]).content
    destino = "legalista" if "legal" in (notas or "").lower() else "productor"
    print(f"🤝 Recepción cede a {destino}: {notas}")
    return Command(goto=destino, update={"notas": notas, "saltos": state["saltos"] + 1})


def legalista(state: EstadoHand) -> Command:
    texto = llm.invoke([
        SystemMessage(content="Legal en Chile. 3 líneas. Si falta un alcance de producto, dilo con la palabra PRODUCTO. No cites RGPD."),
        HumanMessage(content=f"{state['consulta']}\nNotas: {state['notas']}"),
    ]).content
    if "PRODUCTO" in (texto or "") and state["saltos"] < MAX_SALTOS:
        print("🤝 Legal cede a productor")
        return Command(goto="productor", update={"notas": texto, "saltos": state["saltos"] + 1})
    return Command(goto=END, update={"respuesta": texto})


def productor(state: EstadoHand) -> Command:
    texto = llm.invoke([
        SystemMessage(content="Producto. 3 líneas con next step concreto."),
        HumanMessage(content=f"{state['consulta']}\nNotas: {state['notas']}"),
    ]).content
    return Command(goto=END, update={"respuesta": texto})


h = StateGraph(EstadoHand)
h.add_node("recepcion", recepcion)
h.add_node("legalista", legalista)
h.add_node("productor", productor)
h.add_edge(START, "recepcion")
app_hand = h.compile()
mostrar_grafo(app_hand)

out = app_hand.invoke({
    "consulta": "Un apoderado pide borrar las notas de su hijo del Excel del curso.",
    "notas": "",
    "saltos": 0,
    "respuesta": "",
})
print("\nRespuesta final:\n", out["respuesta"])
print("Saltos usados:", out["saltos"], "/ tope", MAX_SALTOS)


### 5. Fan-out (`Send`) — paralelo, no comité

El mismo nodo, N entradas. Útil para revisar 3 cláusulas o 3 CV. El reducer (`operator.add`) junta los parciales. No hace falta un “agente revisor” por ítem.


In [ ]:
class EstadoFan(TypedDict):
    temas: list[str]
    parciales: Annotated[list[str], operator.add]
    resumen: str


def despachar(state: EstadoFan):
    return [Send("analizar", {"tema": t}) for t in state["temas"]]


def analizar(state: dict) -> dict:
    texto = llm.invoke([
        SystemMessage(content="Una línea: riesgo principal de este tema para un colegio."),
        HumanMessage(content=state["tema"]),
    ]).content
    print(f"⚙️ {state['tema']} → {texto}")
    return {"parciales": [f"{state['tema']}: {texto}"]}


def reducir(state: EstadoFan) -> dict:
    texto = llm.invoke([
        SystemMessage(content="Junta los riesgos en 3 viñetas. Sin intro."),
        HumanMessage(content="\n".join(state["parciales"])),
    ]).content
    return {"resumen": texto}


f = StateGraph(EstadoFan)
f.add_node("analizar", analizar)
f.add_node("reducir", reducir)
f.add_conditional_edges(START, despachar, ["analizar"])
f.add_edge("analizar", "reducir")
f.add_edge("reducir", END)
app_fan = f.compile()

out = app_fan.invoke({
    "temas": [
        "notas en Excel compartido",
        "fotos de menores en el Instagram del colegio",
        "un chatbot que responde tareas",
    ],
    "parciales": [],
    "resumen": "",
})
print("\nResumen:\n", out["resumen"])


### 6. Criterio para el proyecto (llena esto en la pauta)

| Pregunta | Tu respuesta |
|---|---|
| ¿Un solo `create_agent` resuelve el enunciado? | |
| Si hay segundo rol: ¿chocan los prompts o solo son pasos? | |
| ¿Supervisor, handoff o `Send`? | |
| ¿Tope de saltos / iteraciones? | |
| ¿Tokens de un agente vs el equipo (aprox.)? | |
| ¿CrewAI aporta un `Process` que el grafo no cubre? | |

**Default del curso:** `create_agent` → supervisor si hace falta → handoff solo si B necesita el hilo → CrewAI si el cliente pide roles con nombre. Python puro para DAGs y negociación.

Cierra con [`4-tendencias-multiagente.md`](4-tendencias-multiagente.md). El Swarm de OpenAI (`Swarm_101.ipynb`) se mira como historia: el handoff de arriba es el patrón vigente.
